# Wang, Romani, Lustig, Leonardo & Pastalkova (2015) on CRCNS hc-5

**Paper.** Y. Wang, S. Romani, B. Lustig, A. Leonardo, E. Pastalkova, "Theta sequences are essential
for internally generated hippocampal firing fields", *Nature Neuroscience* 18, 282-288 (2015),
doi:10.1038/nn.3904.

**Claims.** `claims/wang-2014.yaml`, written from the version of record (main text, figure legends,
Online Methods) and the Supplementary Information (Supplementary Modeling, Supplementary Table 1 and
the legends of Supplementary Figures 1 to 17), checked by a second reader.

**Dataset and version.** CRCNS hc-5, doi:10.6080/K0KS6PHF, read from the public S3 bucket
`crcnsarchive` (`https://crcnsarchive.s3-us-west-2.amazonaws.com/hc-5/`). CRCNS data sets are not
versioned and this one has a single published copy, so the version is recorded as `unversioned`
(docs/archives.md, "CRCNS"); the paper cites no version. The MD5 sums of the files read are printed
below from the data set's own `checksums.md5`, so that a later change of a file can be detected. The
files are 13 `tar.gz` archives, one per session; a `tar.gz` cannot be read at an offset, so each is
streamed from its start and only the members needed are kept in memory. Nothing is written to disk.

**What the data check finds.** The data set holds 13 sessions of one rat (i01) recorded in June 2008:
seven sessions of the left/right alternation task with 15-20 s wheel runs, one home-cage wheel
session and five open-platform sessions. None of them is a muscimol, saline, pre- or post-injection
session, and there is no linear track. Every finding of the paper on recorded data compares sessions
before, under and after medial septum inactivation (or saline), in three rats whose doors opened after
9 or 10 s of wheel running; none of these recordings is in the data set. Those claims are blocked
(`data_missing`). The three claims about the network model use no recorded data and are run here.

| Claim | Tested here? |
|---|---|
| cells-per-recording | blocked, data_missing |
| fig1d-theta-power, fig1e-performance, fig1e-performance-restored, figs1-activity-unchanged | blocked, data_missing |
| fig2c-wheel-field-info, fig2b-wheel-sequence-lost, figs4a-residual-fields-early, fig2d-wheel-rate, fig2e-wheel-theta-sequences | blocked, data_missing |
| fig2h-maze-field-info, fig2i-maze-order-preserved, fig2j-maze-theta-sequences, figs5-compression-lost | blocked, data_missing |
| fig3c-model-theta-builds-fields | tested (simulation) |
| fig3d-model-cues-keep-fields | tested (simulation) |
| figs10-model-dc-no-fields | tested (simulation) |
| fig4b-linear-track-info, fig4b-linear-track-theta-modulation, fig4a-linear-track-remap, figs11-control-fields-stable | blocked, data_missing |
| fig5c-platform-info, fig5b-platform-wall-fields, figs14-no-stable-fields, figs15-higher-speed, figs16-small-platform | blocked, data_missing |
| fig6-repeated-muscimol-low, fig6-repeated-post-normal, fig6-familiar-impervious | blocked, data_missing |

Runtime: about 20 minutes, most of it the full streaming of the 130 GB of archives for the inventory.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import re, time, resource, urllib.request
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
import hc5
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 120)
T0 = time.time()

## Data check

### Files, sizes and checksums

The bucket's listing, the data set's `checksums.md5` and `filelist.txt`, and the session table of
the data descriptor (`docs/crcns-hc5-data-description.pdf`, Table 1, "Description of sessions
included in the data set"), whose labels are compared with the files below.

In [ ]:
listing = urllib.request.urlopen(hc5.BUCKET.rsplit("hc-5/", 1)[0] + "?list-type=2&prefix=hc-5/").read().decode()
objs = re.findall(r"<Key>(.*?)</Key>.*?<Size>(\d+)</Size>", listing)
md5 = dict(reversed(l.split()) for l in hc5.read_text("checksums.md5").splitlines() if l.strip())
rows = [dict(key=k, size_gb=int(s) / 1e9, md5=md5.get(k.replace("hc-5/", ""), "")) for k, s in objs if not k.endswith("/")]
files = pd.DataFrame(rows)
print(files.to_string(index=False))
print("tar.gz files:", sum(files.key.str.endswith(".tar.gz")), " total GB:", round(files.size_gb.sum(), 1))
assert sorted(k.replace("hc-5/", "") for k in files.key if k.endswith(".tar.gz")) == sorted(hc5.FILES)

In [ ]:
# Session labels of the data descriptor (Table 1), copied as metadata to compare with the files
DESCRIPTOR = {
    "i01_maze05.005.tar.gz": "L/R alternation task, 18 min, 15sec wheel runs, several errors",
    "i01_maze06.002.tar.gz": "L/R alternation task, 18 min, 15-17sec wheel runs, good behavior and performance",
    "i01_maze06.005.tar.gz": "L/R alternation task, 20 min, 20sec wheel runs, several errors",
    "i01_maze08.001.tar.gz": "L/R alternation task, 35 min, 15sec wheel runs, several errors",
    "i01_maze08.003.tar.gz": "Home cage: good spontaneous wheel running, some cage exploration, 34min",
    "i01_maze08.004.tar.gz": "L/R alternation task, 24 min, 15sec wheel runs, several errors",
    "i01_maze8.006.tar.gz": "Large 2D platform exploration, 60min (listed as i01_maze08.006.tar.gz)",
    "i01_maze09.002.tar.gz": "Large 2D platform exploration, 38min",
    "i01_maze13.002.tar.gz": "Large 2D platform exploration, 50min",
    "i01_maze13.003.tar.gz": "L/R alternation task, 53 min, 20sec wheel runs, several errors",
    "i01_maze15.001.tar.gz": "Large 2D platform exploration, 60min",
    "i01_maze15.002.tar.gz": "L/R alternation task, 23 min, 20sec wheel runs, several errors",
    "i01_maze16.002.tar.gz": "Large 2D platform exploration, 59min",
}
print("descriptor lists", len(DESCRIPTOR), "sessions; bucket holds", len(hc5.FILES), "tar.gz files")

### Every member of every archive

Each archive is streamed in full (four at a time) to list its members and read the small parameter
files: the `.xml` (recording date, description and notes written by the experimenters), the `.meta`
of the acquisition system, and the `.cat.evt` that records which recordings were concatenated into the
session. Large members are skipped without being kept.

In [ ]:
t = time.time()
with ProcessPoolExecutor(4) as ex:
    INV = {d["file"]: d for d in ex.map(hc5.inventory, hc5.FILES)}
print(f"inventory streamed in {time.time() - t:.0f} s")
for f in hc5.FILES:
    d = INV[f]
    exts = Counter(re.sub(r"^i01_maze\d+[-_.A-Z]*\d{3}", "", n).split(".")[-1] if "." in n else n for n, s in d["members"])
    tot = sum(s for n, s in d["members"]) / 1e9
    print(f"\n#### {f}: {len(d['members'])} members, {tot:.1f} GB uncompressed; streamed in {d['seconds']:.0f} s")
    print("   largest:", ", ".join(f"{n} ({s/1e9:.2f} GB)" for n, s in sorted(d["members"], key=lambda x: -x[1])[:4]))
    print("   processed file:", [n for n, s in d["members"] if "_BehavElectrData" in n])
    print("   xml date:", d["xml_date"], "| channels:", d["xml_nchannels"], "| notes:", repr(d["xml_notes"]))
    print("   xml description:", repr(d["xml_description"]))
    for k, v in d["texts"].items():
        if k.endswith(".cat.evt"):
            print("   ", k, "->", v.strip().replace("\n", " | "))
        if k.endswith(".meta"):
            keep = [l for l in v.splitlines() if l.strip().startswith(("System", "Filename", "Recording start date", "Number of recorded"))]
            print("   ", k, "->", " | ".join(l.strip() for l in keep))

In [ ]:
# one complete member list, for the record
d = INV["i01_maze08.001.tar.gz"]
print("\n".join(f"{s:>14} {n}" for n, s in d["members"]))

### The processed file of each session

Each session holds one processed MATLAB file with spikes, clusters, tracking, laps and one LFP
channel at 1,250 Hz (`*_BehavElectrData.mat`, MATLAB 5, in the ten alternation and home-cage
archives; `*_BehavElectrDataLFP.mat`, MATLAB 7.3, in three platform archives). Its full layout is
listed for one session of each kind, and a summary is computed for every session
(`hc5.summarize_session`).

In [ ]:
import io, scipy.io as sio, h5py
for f in ["i01_maze08.001.tar.gz", "i01_maze09.002.tar.gz"]:
    name, b = hc5.processed_bytes(f)
    print(f"\n#### {f}: {name}, {len(b)/1e6:.0f} MB; header: {b[:80].decode(errors='replace').strip()}")
    if b[:19] == b"MATLAB 7.3 MAT-file":
        h = h5py.File(io.BytesIO(b), "r")
        def show(n, o):
            if isinstance(o, h5py.Dataset) and not n.startswith("#refs#"):
                print(f"   {n}: {o.shape} {o.attrs.get('MATLAB_class', b'').decode()}")
        h.visititems(show); h.close()
    else:
        d = sio.loadmat(io.BytesIO(b), squeeze_me=True, struct_as_record=False)
        for k, v in d.items():
            if k.startswith("__"):
                continue
            for fn in v._fieldnames:
                a = np.asarray(getattr(v, fn))
                val = f" = {a.item()!r}" if a.size == 1 else ""
                print(f"   {k}.{fn}: {a.shape} {a.dtype}{val}")
    del b

In [ ]:
t = time.time()
with ProcessPoolExecutor(3) as ex:
    SUM = {d["file"]: d for d in ex.map(hc5.summarize_session, hc5.FILES)}
print(f"processed files read in {time.time() - t:.0f} s; peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.1f} GB")
rows = []
for f in hc5.FILES:
    s = SUM[f]
    rows.append(dict(
        file=f.replace(".tar.gz", ""), task=s["task"], layout=s["layout"], minutes=round(s["minutes"], 1),
        xml_date=INV[f]["xml_date"], clusters=s["n_clusters"], interneurons=s["n_interneurons"], shanks=s["n_shanks"],
        spikes=s["n_spikes"], laps=s.get("n_laps"), behav_type=s.get("behav_type_counts"),
        wheel_runs=len(s["wheel_run_s"]),
        wheel_run_s_median=float(np.median(s["wheel_run_s"])) if s["wheel_run_s"] else None,
        wheel_run_s_range=(min(s["wheel_run_s"]), max(s["wheel_run_s"])) if s["wheel_run_s"] else None))
SESS = pd.DataFrame(rows)
print(SESS.to_string(index=False))
print("\nprocessed-file headers:")
for f in hc5.FILES:
    print("  ", f, "|", SUM[f]["member"], "|", SUM[f]["header"][:75])

In [ ]:
print("Ranges of the variables used in the check (file units):")
for f in hc5.FILES:
    s = SUM[f]
    print(f"  {f}: speed {s['speed_range'][0]:.1f} to {s['speed_range'][1]:.0f} ({'mm/s' if s['layout']=='v7.3' else 'Track.speed'}),"
          f" x {s['x_range'][0]:.0f} to {s['x_range'][1]:.0f}, y {s['y_range'][0]:.0f} to {s['y_range'][1]:.0f},"
          f" wheel speed max {s.get('wheel_speed_max', float('nan')):.0f}, spike samples {s['res_range'][0]} to {s['res_range'][1]} of {s['res_range'][2]}")
print("\nSpike table against tracking table (Spike.res taken as a 1-based index, Track row res - 1):")
for f in hc5.FILES:
    s = SUM[f]
    px, ph = s["spike_x_matches_track"], s["spike_phase_matches_track"]
    print(f"  {f}: position at spike equals Track position in " + (f"{100*px:.2f}%" if px is not None else "n/a") +
          " of spikes; theta phase at spike equals Track theta phase in " + (f"{100*ph:.2f}%" if ph is not None else "n/a") +
          f"; lengths of the spike fields {s['spike_field_lengths']}")

### Theta in every session, and the clock of LFP and behaviour

Medial septum inactivation with muscimol reduces hippocampal theta by about 80% (the paper's
Figure 1d) and the paper's muscimol sessions were recorded about 20 min after an injection. If any
session of the data set were a muscimol session, its theta during running would be weak. For every
session: the multitaper (NW = 4) power spectrum of the processed file's LFP channel in 2-s windows
during running (on the wheel, or speed smoothed over 1 s above its 75th percentile) and during
stillness; the theta (6-10 Hz) peak, its frequency, the theta/delta ratio, and the same in each third
of the session. The clock check correlates, over 1-s windows, the LFP's log theta/delta with the
fraction of the window spent running, with the running series shifted by -30 to +30 s: if the LFP
and the behaviour share a clock, the correlation peaks near zero lag (theta follows running within a
second). The LFP's units differ between sessions (the `Track.eeg` scaling differs), so powers are
compared within a session; the ratios can be compared across sessions.

In [ ]:
rows = []
for f in hc5.FILES:
    s = SUM[f]
    r, st, xc = s["theta_run"], s["theta_still"], s["clock_xcorr"]
    thirds = [x["theta_delta"] if x else np.nan for x in s["theta_run_thirds"]]
    rows.append(dict(file=f.replace(".tar.gz", ""), task=s["task"],
                     run_windows=r["n_windows"] if r else 0,
                     run_theta_hz=r["theta_peak_hz"] if r else np.nan,
                     run_theta_delta=round(r["theta_delta"], 2) if r else np.nan,
                     still_theta_delta=round(st["theta_delta"], 2) if st else np.nan,
                     run_over_still_theta_power=round(r["theta_peak_power"] / st["theta_peak_power"], 2) if (r and st) else np.nan,
                     thirds_theta_delta=tuple(round(x, 2) for x in thirds),
                     clock_r_lag0=round(xc["r_at_0"], 3), clock_best_lag_s=xc["best_lag_s"], clock_r_best=round(xc["r_best"], 3),
                     clock_r_lag_pm30=round(xc["r_at_pm30"], 3)))
TH = pd.DataFrame(rows)
print(TH.to_string(index=False))
ok = TH.clock_best_lag_s.abs() <= 2
print("\nclock check: correlation of theta/delta with running peaks within 2 s of zero lag in", int(ok.sum()), "of", len(TH),
      "sessions; not in:", TH.file[~ok].tolist())
print("theta peak frequency during running, range over sessions:", TH.run_theta_hz.min(), "to", TH.run_theta_hz.max(), "Hz")
print("theta/delta during running, range over sessions:", TH.run_theta_delta.min(), "to", TH.run_theta_delta.max())
print("theta/delta during running over stillness, range:", round((TH.run_theta_delta / TH.still_theta_delta).min(), 2), "to",
      round((TH.run_theta_delta / TH.still_theta_delta).max(), 2))
print("lowest theta/delta in a third of a session during running:", np.nanmin(np.concatenate([np.array(x, float) for x in TH.thirds_theta_delta])))

### What the claims need, against what the data set holds

The claims file's opening comment describes the paper's data: three rats (Long-Evans, two 64-channel
probes in left and right dorsal CA1, a guide cannula over the medial septum) whose maze doors opened
after 9 (rat 1) or 10 s (rats 2, 3) of wheel running; alternation recordings made of a pre session, a
muscimol session about 20 min after injection and a post session about 4 h later (15 muscimol and 7
saline recordings); a novel linear track (muscimol and control); large novel and familiar platforms
under muscimol or saline, with post sessions about 6 h later; a smaller platform; and repeated
muscimol visits of the same platform over days 1 to 3 and after familiarization.

In [ ]:
alt = SESS[SESS.task == "LRalternation"]
print("alternation sessions:", len(alt), "| median wheel-run durations per session (s):", alt.wheel_run_s_median.round(1).tolist())
allruns = np.concatenate([SUM[f]["wheel_run_s"] for f in hc5.FILES if SUM[f]["task"] == "LRalternation"])
print(f"all alternation wheel runs: n = {len(allruns)}, median {np.median(allruns):.1f} s, 5th-95th percentile "
      f"{np.percentile(allruns, 5):.1f}-{np.percentile(allruns, 95):.1f} s, shortest {allruns.min():.1f} s")
print("recording dates (xml):", sorted({INV[f]['xml_date'] for f in hc5.FILES}))
words = re.compile(r"muscimol|musc|saline|inject|septum|\bMS\b|linear|track|novel|familiar", re.I)
hits = {f: words.findall((INV[f]["xml_description"] or "") + " " + (INV[f]["xml_notes"] or "")) for f in hc5.FILES}
print("words about injection, track or novelty in the xml descriptions and notes:", {k: v for k, v in hits.items() if v} or "none")
names = [n for f in hc5.FILES for n, s in INV[f]["members"]]
print("member names mentioning muscimol, saline, an injection or a linear track (not 'linearized' or 'SyncTrackCoord'):",
      [n for n in names if re.search(r"musc|saline|inject|linear_?track|lintrack", n, re.I)] or "none")
print("sessions concatenated in a .cat.evt (M = maze, S = sleep):",
      {k: re.findall(r"beginning of (\S+)", v) for f in hc5.FILES for k, v in INV[f]["texts"].items() if k.endswith(".cat.evt")})

**Checks of the description against the data.**

- *Sessions and sizes.* The bucket holds the 13 archives of the descriptor's Table 1; one is named
  `i01_maze8.006.tar.gz` where the descriptor lists `i01_maze08.006.tar.gz`. Durations agree with
  the descriptor within 2 min, except `i01_maze13.003` (60.2 min of data against 53 min in the
  descriptor and its `.xml`).
- *Task labels.* For `i01_maze13.002` and `i01_maze15.001` the descriptor and the `.xml` say
  large open-field exploration, but the processed file's task label is `WheelHomeCage`. For
  `i01_maze08.001` the processed file and the descriptor say alternation task (28 laps), but the
  `.xml` description says "home cage whl running". The data agree with the descriptor in both cases
  (laps and trial outcomes in `i01_maze08.001`; no laps in the other two).
- *Processed-file layout.* The descriptor's field list (`Spike.xMM`, `Track.xMM`, `Laps.lapID`,
  `Clu.ccgVal`) matches the MATLAB 7.3 files of the platform sessions, and the descriptor names them
  `*_BehavElectrDataLFP.mat`; the ten MATLAB 5 files (`*_BehavElectrData.mat`) use other field names
  (`Spike.X`, `Track.X`, `Laps.NLap`, `Par`), and these differ between sessions too. `Spike.res` is a
  1-based (MATLAB) index into the tracking table, which the descriptor does not say; with that
  reading the position stored at each spike equals the tracking table's for every spike of every
  session. The theta phase stored at each spike equals `Track`'s theta phase only in some sessions
  (printed above), so the two are different phase estimates in the others; in three sessions
  (`i01_maze13.002`, `i01_maze13.003`, `i01_maze15.001`) the spike theta-phase array is shorter than
  the spike-time array, so it cannot be matched to the spikes.
- *Units.* The processed files do not state the units of position and speed, and they differ
  between sessions: in the MATLAB 5 files x reaches about 1,300 in five alternation sessions but
  460 in `i01_maze13.003` and 530-570 in the home-cage and two platform sessions, and the wheel
  speed reaches about 1,000 in most alternation sessions but 18,400 in `i01_maze13.003` (ranges
  printed above). The MATLAB 7.3 files give millimetres (`xMM`, `speed_MMsec`).
- *Channels.* The descriptor gives 125 neural channels plus three event channels (128 or 129); 12 of
  13 sessions have 102 channels in their `.xml`, one (`i01_maze05.005`) 128.
- *Dates.* The `.xml` dates are 1 to 19 June 2008; the `.meta` files of the platform sessions give
  "Recording start date = Sun Apr 22 2012", which is after the recordings and appears to be the date
  the files were converted.
- *Clock.* In 12 of 13 sessions the correlation between the LFP's theta/delta and running peaks
  within a second of zero lag (printed above): the LFP and the behaviour are consistent with one
  clock, to about a second. In `i01_maze8.006` theta barely differs between running and stillness
  and the correlation is flat, so the clock could not be established there. The spikes share the
  tracking table's sample index (position check above).
- *No injection sessions.* Theta during running has its peak at 6.5 to 8 Hz in every session and its
  theta/delta is above stillness in every session (ratios printed above), least in `i01_maze8.006`
  (about 1.1, where the clock check also fails) and most in `i01_maze09.002` (about 9.5). No session shows the loss of theta that muscimol causes in the paper (81% less);
  the lowest theta/delta in a third of a session is printed above.
  No description, note or member name mentions muscimol, saline, an injection, the septum, a linear
  track or novelty. The `MS` in the file names is not the medial septum: the `.cat.evt` files show
  that it marks a maze recording (`M`) concatenated with the following sleep recording (`S`). Wheel
  runs last a median of about 20 s (none as short as the paper's 9-10 s), and the probes include a
  right entorhinal shank group, which the paper's rats did not have. The data are consistent with
  rat i01 of Pastalkova et al. (2008) and Mizuseki et al. (2009), the other two papers the descriptor
  names, recorded without any injection.

## Claims on recorded data: all blocked

Every claim of the paper on recorded data compares pre-injection, muscimol, post-injection or saline
sessions, or uses a linear track, novel and familiar platforms under muscimol or saline, or the paper's
three rats; none of these recordings is in the data set (checks above). They are blocked as
`data_missing`. The alternation sessions of rat i01 are uninjected recordings of a different rat,
with 15-20 s wheel runs, so they cannot stand for the paper's pre or post sessions (rats 1 to 3 ran
9 or 10 s), and `cells-per-recording` is blocked too. No claim depends on these sessions, so no
analysis of them is run here.

## The network model (Supplementary Modeling)

The three model claims use no recorded data. The rate model of Supplementary Modeling (equations 1-3,
the network parameters listed there and the currents of Supplementary Table 1) is integrated as
`hc5.simulate`, and the measures fixed in the claims file are computed by `hc5.model_measures` (the
helper's docstring gives the source of every parameter). Choices the paper leaves open, as listed in
the criterion of `fig3c-model-theta-builds-fields`: 10 s simulated, Euler step 0.1 ms, uniform low
initial rates, five seeds, first second discarded. The noise term is written eta(t); it is read as
independent noise in each unit, because the paper's Figure 3c,d show the no-theta activity scattered
independently over neurons and the Results say that without theta "neurons in the network were
activated randomly". This is an assumption; the reading of one noise process shared by all units is
run as a variant and shown next to it.

Measures (per simulation, after the first second):
`pop_min_over_max`, the minimum over the maximum of the population rate within each cycle (near 0
when activity is confined to theta cycles); `frac_cycles_forward`, the share of cycles in which the
bump moves forward (in the direction of its drift) between the first and last time the population
rate is at half its cycle peak; `sweep_rad`, the median size of that displacement; `drift_rad_s`,
the bump's mean travel; `bump_concentration`, how concentrated the activity is on the ring (1 at one
place, near 0 spread out); `field_peak_cycles`, the median over units of the time the unit's rate,
smoothed over one cycle, stays above half its peak around the peak, in theta cycles; and
`field_all_cycles`, the same over every run above half the peak. Theta sequences are counted as
present when activity is confined to cycles (`pop_min_over_max` < 0.1) and the bump moves forward in
at least 90% of them; these two thresholds are set here, before the model was run with them, and are
part of the deviation the criterion records.

In [ ]:
CONDS = {
    # name: (simulate kwargs, cycle frequency for the measures, cue speed or None)
    "theta/no cues": (dict(I=2, I_theta=30, I_L=0, f_theta=7), 7, None),
    "no theta/no cues": (dict(I=2, I_theta=0, I_L=0), 7, None),
}
for lab, v in [("v/2", np.pi / 5), ("v", 2 * np.pi / 5), ("2v", 4 * np.pi / 5)]:
    CONDS[f"theta/cues {lab}"] = (dict(I=2, I_theta=30, I_L=5, f_theta=8, cue_speed=v), 8, v)
    CONDS[f"no theta/cues {lab}"] = (dict(I=2, I_theta=0, I_L=5, cue_speed=v), 8, v)
for I in (4, 6, 10):
    CONDS[f"no theta/no cues I={I}"] = (dict(I=I, I_theta=0, I_L=0), 7, None)
SEEDS = range(5)
NOISES = ("unit", "common")

def run_one(job):
    name, noise, seed = job
    kw, fc, cs = CONDS[name]
    R = hc5.simulate(**kw, seed=seed, noise=noise)
    return dict(condition=name, noise=noise, seed=seed, **hc5.model_measures(R, fc, cue_speed=cs))

jobs = [(c, n, s) for c in CONDS for n in NOISES for s in SEEDS]
t = time.time()
with ProcessPoolExecutor(3) as ex:
    MODEL = pd.DataFrame(list(ex.map(run_one, jobs)))
print(f"{len(jobs)} simulations in {time.time() - t:.0f} s")
cols = ["pop_min_over_max", "frac_cycles_forward", "sweep_rad", "drift_rad_s", "bump_concentration",
        "field_peak_s", "field_peak_cycles", "field_all_cycles", "mean_rate_hz", "drift_over_cue_speed"]
MEAN = MODEL.groupby(["noise", "condition"], sort=False)[cols].mean()
RANGE = MODEL.groupby(["noise", "condition"], sort=False)[["field_peak_cycles", "drift_rad_s"]].agg(["min", "max"])

def seq_present(r):
    return (r.pop_min_over_max < 0.1) & (r.frac_cycles_forward >= 0.9)
MODEL["sequences"] = seq_present(MODEL)
print(MEAN.round(3).to_string())

### fig3c-model-theta-builds-fields

> In a recurrent network model with asymmetric connections, short-term depression and facilitation,
> and theta input but no sensory cues, a bump of activity travels in each theta cycle (theta
> sequences) and neurons fire over several cycles (firing fields); removing the theta input disrupts
> both.

Criterion: reproduced when, with theta, the bump moves within each cycle and units fire over several
consecutive cycles (fields longer than three cycles), and without theta neither holds.

In [ ]:
for noise in NOISES:
    print(f"--- noise read as '{noise}'" + (" (adopted)" if noise == "unit" else " (variant)"))
    for c in ["theta/no cues", "no theta/no cues"]:
        sub = MODEL[(MODEL.noise == noise) & (MODEL.condition == c)]
        m = MEAN.loc[(noise, c)]
        print(f"  {c:18s}: theta sequences present in {int(sub.sequences.sum())}/{len(sub)} seeds; "
              f"pop min/max {m.pop_min_over_max:.3f}; forward in {100*m.frac_cycles_forward:.0f}% of cycles, sweep {m.sweep_rad:.2f} rad; "
              f"bump concentration {m.bump_concentration:.2f}; drift {m.drift_rad_s:.2f} rad/s")
        print(f"  {'':18s}  field around the peak {m.field_peak_s:.3f} s = {m.field_peak_cycles:.2f} cycles of 7 Hz "
              f"(seeds {sub.field_peak_cycles.min():.2f}-{sub.field_peak_cycles.max():.2f}); every run {m.field_all_cycles:.2f} cycles; "
              f"fields > 3 cycles in {int((sub.field_peak_cycles > 3).sum())}/{len(sub)} seeds")
    th = MEAN.loc[(noise, "theta/no cues")]; no = MEAN.loc[(noise, "no theta/no cues")]
    print(f"  field ratio with/without theta: {th.field_peak_s / no.field_peak_s:.2f} (around the peak), "
          f"{th.field_all_cycles / no.field_all_cycles:.2f} (every run)")
    print(f"  threshold for 'several': fields > 3 cycles = {3/7:.3f} s; theta fields reach {th.field_peak_s:.3f} s")

In [ ]:
# How the field length with theta depends on the open simulation choices (adopted noise reading):
# the Euler step (0.1 ms against 0.05 ms) and the duration (10 s against 30 s)
def run_sens(job):
    dt, T, seed = job
    R = hc5.simulate(I=2, I_theta=30, I_L=0, f_theta=7, seed=seed, noise="unit", dt=dt, T=T)
    return dict(dt_ms=dt * 1e3, T_s=T, seed=seed, **hc5.model_measures(R, 7))
with ProcessPoolExecutor(3) as ex:
    SENS = pd.DataFrame(list(ex.map(run_sens, [(dt, T, sd) for dt, T in [(1e-4, 10.0), (5e-5, 10.0), (1e-4, 30.0)] for sd in SEEDS])))
g = SENS.groupby(["dt_ms", "T_s"])
print(pd.DataFrame({"field_peak_cycles_mean": g.field_peak_cycles.mean(), "min": g.field_peak_cycles.min(),
                    "max": g.field_peak_cycles.max(), "seeds_above_3": g.field_peak_cycles.apply(lambda x: int((x > 3).sum())),
                    "field_all_cycles_mean": g.field_all_cycles.mean(), "frac_cycles_forward": g.frac_cycles_forward.mean(),
                    "drift_rad_s": g.drift_rad_s.mean()}).round(3).to_string())

**Result: partially reproduced, resting on the model choices stated above.** With theta, activity is
confined to theta cycles (population minimum over maximum 0.002) and the bump moves forward within
98% of cycles (median 0.45 rad), so theta sequences are present in all five seeds; without theta they
are absent in all five, and the activity is scattered over the ring (concentration 0.19 against 0.84
with theta). Fields with theta last 3.02 theta cycles on average around the peak (2.96 to 3.12 over
seeds, two of five above three), 2.98 counting every run, and 2.88 with the shared-noise variant;
with a 0.05 ms step or 30 s simulated they stay between 2.96 and 3.04. Without theta they last 1.33
cycles around the peak (0.65 counting every run). So theta lengthens the fields two- to fivefold, but
their length sits at the criterion's reading of "several" (more than three cycles) and does not
clearly exceed it: the field part holds at the threshold only.

### fig3d-model-cues-keep-fields

> With a slowly moving familiar cue input added to the model, removing theta still disrupts theta
> sequences, but firing fields are maintained when the cue input is strong enough.

Criterion: with the cue currents of Supplementary Table 1 (theta/cues I_theta = 30, I_L = 5,
f = 8 Hz; no theta/cues I_theta = 0, I_L = 5), and cue speeds 2pi/5 rad/s (v) and half and twice it.
Reproduced when, under a cue speed reading, without theta the within-cycle sequences are gone but
units keep fields that follow the cue. Fields are judged as in fig3c (more than three cycles, here of
8 Hz), and following the cue as the bump's mean travel within 20% of the cue's speed (set here).

In [ ]:
for noise in NOISES:
    print(f"--- noise read as '{noise}'" + (" (adopted)" if noise == "unit" else " (variant)"))
    for lab in ["v/2", "v", "2v"]:
        for c in [f"theta/cues {lab}", f"no theta/cues {lab}"]:
            sub = MODEL[(MODEL.noise == noise) & (MODEL.condition == c)]
            m = MEAN.loc[(noise, c)]
            follows = abs(m.drift_over_cue_speed - 1) <= 0.2
            print(f"  {c:18s}: sequences in {int(sub.sequences.sum())}/{len(sub)} seeds (pop min/max {m.pop_min_over_max:.3f}, "
                  f"forward {100*m.frac_cycles_forward:.0f}%); bump travel / cue speed {m.drift_over_cue_speed:.2f} ({'follows' if follows else 'does not follow'}); "
                  f"field {m.field_peak_cycles:.2f} cycles around peak (seeds {sub.field_peak_cycles.min():.2f}-{sub.field_peak_cycles.max():.2f}), "
                  f"{m.field_all_cycles:.2f} every run; > 3 cycles in {int((sub.field_peak_cycles > 3).sum())}/{len(sub)} seeds")

**Result: reproduced under the adopted noise reading, not under the variant.** With independent
noise per unit, removing theta removes the theta-cycle structure at every cue speed (no seed has
sequences; population minimum over maximum 0.75 to 0.79 against 0.004 to 0.005 with theta), while
the bump keeps following the cue (travel 1.00 times the cue speed) and units keep fields around their
peak of 8.34, 5.90 and 3.32 cycles of 8 Hz at v/2, v and 2v, longer than three cycles in every seed.
Counted over every run above half the peak they are 1.33, 4.48 and 3.23 cycles, so they hold at v and
2v under that count. With one noise process shared by all units, fields without theta last 2.78,
3.00 and 2.28 cycles around the peak, not longer than three cycles, so the outcome rests on the noise
reading, taken from the paper's Figure 3c,d.

### figs10-model-dc-no-fields

> Without theta, raising the nonspecific input of the model produces fast sequential activity, not
> firing fields.

Criterion: without theta, with I raised (readings I = 4, 6 and 10). Reproduced when, at the raised
input, the bump moves through the network faster than the field sequence speed with theta and units do
not fire over durations as long as the fields with theta.

In [ ]:
for noise in NOISES:
    print(f"--- noise read as '{noise}'" + (" (adopted)" if noise == "unit" else " (variant)"))
    ref = MEAN.loc[(noise, "theta/no cues")]
    print(f"  reference, theta/no cues: drift {ref.drift_rad_s:.2f} rad/s, field {ref.field_peak_s:.3f} s")
    for c in ["no theta/no cues", "no theta/no cues I=4", "no theta/no cues I=6", "no theta/no cues I=10"]:
        sub = MODEL[(MODEL.noise == noise) & (MODEL.condition == c)]
        m = MEAN.loc[(noise, c)]
        faster = int((sub.drift_rad_s.abs() > ref.drift_rad_s).sum())
        shorter = int((sub.field_peak_s < ref.field_peak_s).sum())
        print(f"  {c:22s}: drift {m.drift_rad_s:6.2f} rad/s (seeds {sub.drift_rad_s.min():.2f} to {sub.drift_rad_s.max():.2f}; "
              f"faster than with theta in {faster}/{len(sub)}), concentration {m.bump_concentration:.2f}, "
              f"field {m.field_peak_s:.3f} s (shorter in {shorter}/{len(sub)})")

**Result: reproduced at I = 6 and I = 10, not at I = 4, resting on the reading of "high".** With
theta the field sequence travels at 2.99 rad/s and fields last 0.43 s (adopted noise reading). Without
theta, at I = 10 the bump forms (concentration 0.88) and travels at 14.8 rad/s, and at I = 6 at 6.4
rad/s, faster than with theta in every seed, while fields last 0.15 and 0.22 s around the peak,
shorter in every seed. At I = 4 the bump is weak (concentration 0.30) and travels at 2.6 rad/s on
average, faster than with theta in only some seeds (counts printed above). The shared-noise variant gives a faster bump at all three levels.
Supplementary Figure 10c plots the DC input from 1 to 10, and 10b's "high" level shows a bump
crossing the network in a fraction of a second, so I = 10 is the reading adopted.

## Summary

| Claim | Outcome |
|---|---|
| 26 claims on recorded data (Figures 1, 2, 4, 5, 6, Supplementary Figures 1, 4, 5, 8, 11-16, and the cell counts) | blocked, data_missing |
| fig3c-model-theta-builds-fields | partially reproduced (assumed) |
| fig3d-model-cues-keep-fields | reproduced (assumed) |
| figs10-model-dc-no-fields | reproduced (assumed) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

**How the analysis differs from the paper's.** No recorded data were analysed for a claim, because
the data set holds none of the recordings the claims rest on. The model was integrated from the
equations and parameters of Supplementary Modeling, with the duration, step, initial state, seeds and
the structure of the noise chosen here (the noise from the paper's Figure 3c,d); the measures of
sequences and fields are the claims file's, since the paper gives none.

**What was found about the data set.** CRCNS hc-5 holds 13 sessions of one rat (i01) recorded in
June 2008: seven alternation sessions with 15-20 s wheel runs, one home-cage wheel session and five
open-platform sessions, with spikes, clusters, tracking, one LFP channel and raw data. No session is a
muscimol, saline, pre- or post-injection session, there is no linear track, and the rat is not one of
the paper's three rats, whose wheel runs were 9 or 10 s; the data set lists the paper among three that
used "some of these data" but holds none of the data its findings rest on. The `MS` of the file names
marks concatenated maze and sleep recordings, not the medial septum. Two sessions described as
platform exploration carry the task label `WheelHomeCage` in their processed files, one alternation
session's `.xml` describes home-cage running, the descriptor's field list fits only the three MATLAB
7.3 files, and `Spike.res` is a 1-based index that the descriptor does not describe.

In [ ]:
print(f"notebook runtime {(time.time() - T0)/60:.1f} min; peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.1f} GB")